In [ ]:
# alife is a local package; MP4 encoding needs a ffmpeg binary.
# pip install --quiet -e . imageio imageio-ffmpeg

In [ ]:
import os
import pathlib

# Headless rendering: pygame must not try to open a window inside the dev container.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

import imageio.v2 as imageio
from IPython.display import Video
from pettingzoo.utils.wrappers import BaseParallelWrapper

from alife import World

In [ ]:
def save_video(frames, path, fps):
    """Encode a list of RGB frames into an MP4 file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    imageio.mimwrite(path, frames, fps=fps)


class RecordVideoParallel(BaseParallelWrapper):
    """Record an episode of a parallel environment as an MP4 file."""

    def __init__(self, env, video_folder="videos", name_prefix="rl-video", fps=4):
        super().__init__(env)
        self.video_path = pathlib.Path(video_folder) / f"{name_prefix}.mp4"
        self.fps = fps
        self.frames = []

    def reset(self, seed=None, options=None):
        self.frames = []
        return super().reset(seed=seed, options=options)

    def step(self, actions):
        result = super().step(actions)
        self.frames.append(self.render())
        return result

    def close(self):
        super().close()
        save_video(self.frames, self.video_path, self.fps)

In [ ]:
MAP_NAME = "2025"   # only "2025" and "fairy_ring" ship with the package
N_AGENTS = 4
MAX_STEPS = 200     # alife episodes are eternal: env.agents never becomes empty

env = RecordVideoParallel(
    World(render_mode="rgb_array"),
    name_prefix="alife",
    fps=World.metadata["render_fps"],
)

# alife needs an explicit map name (the default one does not exist)
observations, infos = env.reset(seed=42, options={"map_name": MAP_NAME})

# Unlike most PettingZoo environments, agents are added *after* reset()
for i in range(N_AGENTS):
    env.add_agent(f"agent_{i}")

print(f"Agents: {env.agents}")

In [ ]:
for _ in range(MAX_STEPS):
    # this is where you would insert your policy
    actions = {agent: env.action_space(agent).sample() for agent in env.agents}

    observations, rewards, terminations, truncations, infos = env.step(actions)
env.close()

Video(str(env.video_path), embed=True)